In [0]:
silver_df = spark.table("mvn_db_catalog.silver.health_claims")

In [0]:
silver_df.show()

In [0]:
from pyspark.sql.functions import count,sum

provider_df = (
    silver_df
    .groupBy("ProviderID")
    .agg(
        count("ClaimID").alias("TotalClaims"),
        sum("ClaimAmount").alias("TotalAmount")
    )
)

In [0]:
provider_df.show()

In [0]:
gold_provider_path = "abfss://gold@mvnazureproject.dfs.core.windows.net/provider_summary"

(
    provider_df.write
    .format("delta")
    .mode("overwrite")
    .save(gold_provider_path)
)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS mvn_db_catalog.gold.provider_summary
USING DELTA
LOCATION 'abfss://gold@mvnazureproject.dfs.core.windows.net/provider_summary';

In [0]:
%sql
SELECT *
FROM mvn_db_catalog.gold.provider_summary;

In [0]:
hospital_df = (
    silver_df
    .groupBy("Hospital")
    .agg(
        count("ClaimID").alias("TotalClaims"),
        sum("ClaimAmount").alias("TotalAmount")
    )
)

In [0]:
hospital_df.write \
.format("delta") \
.mode("overwrite") \
.save("abfss://gold@mvnazureproject.dfs.core.windows.net/hospital_summary")

In [0]:
%sql
CREATE TABLE mvn_db_catalog.gold.hospital_summary
USING DELTA
LOCATION 'abfss://gold@mvnazureproject.dfs.core.windows.net/hospital_summary';

In [0]:
status_df = (
    silver_df
    .groupBy("Status")
    .count()
)

In [0]:
from pyspark.sql.functions import count,sum

daily_df = (
    silver_df
    .groupBy("ClaimDate")
    .agg(
        count("ClaimID").alias("TotalClaims"),
        sum("ClaimAmount").alias("TotalAmount")
    )
)

In [0]:
status_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@mvnazureproject.dfs.core.windows.net/status_summary")

In [0]:
daily_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@mvnazureproject.dfs.core.windows.net/daily_claim_summary")

CREATING DIMENTIONS TABLS AND FACT TABLES IN GOLD LAYER.    

In [0]:
silver_df = spark.table("mvn_db_catalog.silver.health_claims")

In [0]:
fact_claims_df = silver_df.select(
    "ClaimID",
    "MemberID",
    "ProviderID",
    "Hospital",
    "ClaimAmount",
    "ClaimDate",
    "Status"
)

In [0]:
gold_fact_path = "abfss://gold@mvnazureproject.dfs.core.windows.net/fact_claims"

(
    fact_claims_df.write
        .format("delta")
        .mode("overwrite")
        .save(gold_fact_path)
)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS mvn_db_catalog.gold.fact_claims
USING DELTA
LOCATION 'abfss://gold@mvnazureproject.dfs.core.windows.net/fact_claims';

In [0]:
%sql
SELECT * FROM mvn_db_catalog.gold.fact_claims;

In [0]:
# uild Provider Dimension
dim_provider_df = (
    silver_df
    .select(
        "ProviderID",
        "Hospital"
    )
    .dropDuplicates()
)

In [0]:
dim_provider_df.write \
.format("delta") \
.mode("overwrite") \
.save("abfss://gold@mvnazureproject.dfs.core.windows.net/dim_provider")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS mvn_db_catalog.gold.dim_provider
USING DELTA
LOCATION 'abfss://gold@mvnazureproject.dfs.core.windows.net/dim_provider';

In [0]:
dim_member_df = (
    silver_df
    .select("MemberID")
    .dropDuplicates()
)

In [0]:
dim_member_df.write \
.format("delta") \
.mode("overwrite") \
.save("abfss://gold@mvnazureproject.dfs.core.windows.net/dim_member")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS mvn_db_catalog.gold.dim_member
USING DELTA
LOCATION 'abfss://gold@mvnazureproject.dfs.core.windows.net/dim_member';